# 03 — Dataset Processing

This notebook covers:
- Building an `InternalsDataset` from a list of items
- Attaching task-specific `properties` to each instance
- Running the full dataset and accessing `InternalsRecord` objects
- Dumping to disk with `dump()` and reloading with `load()`
- Optional: saving full hidden states and attention matrices

In [ ]:
import internals_extraction
from internals_extraction import (
    InternalsDataset, InternalsInstance, TextSpan, SpanSpec,
    dump, load,
)
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import tempfile, os, json

MODEL     = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()
print("Ready")

## 1. Define the dataset

Each `InternalsInstance` bundles:
- `text` — the input string
- `properties` — arbitrary task metadata (labels, IDs, split name, …)
- `spans` — optional named token regions

In [ ]:
ITEMS = [
    # Factual relation probing
    {"text": "The Eiffel Tower is located in Paris.",
     "label": "location", "subject": "The Eiffel Tower", "object": "Paris"},
    {"text": "Albert Einstein was born in Ulm.",
     "label": "birthplace", "subject": "Albert Einstein", "object": "Ulm"},
    {"text": "Python was created by Guido van Rossum.",
     "label": "creator", "subject": "Python", "object": "Guido van Rossum"},
    {"text": "The Amazon River flows through Brazil.",
     "label": "location", "subject": "The Amazon River", "object": "Brazil"},
    {"text": "Shakespeare wrote Hamlet.",
     "label": "creator", "subject": "Shakespeare", "object": "Hamlet"},
]

instances = [
    InternalsInstance(
        text=item["text"],
        properties={k: v for k, v in item.items() if k != "text"},
        spans=[
            TextSpan(item["subject"], label="subject"),
            TextSpan(item["object"],  label="object"),
        ],
    )
    for item in ITEMS
]

dataset = InternalsDataset(instances)
print(f"Dataset: {len(dataset)} instances")
print(f"Example instance: {dataset[0]}")

## 2. Run the dataset

In [ ]:
records = dataset.run(
    model,
    tokenizer,
    generate_kwargs={"max_new_tokens": 1},   # we care about the forward pass, not output
    verbose=True,
)

print(f"\n{len(records)} records collected")

## 3. Inspect InternalsRecord

In [ ]:
rec = records[0]
print(rec)
print()
print("properties:        ", rec.properties)
print("resolved_spans:    ", rec.resolved_spans)
print("run:               ", rec.run)
print()
print("input_hs_mean:     ", rec.run.input_hidden_states_mean.shape)
print("output_hs_mean:    ", rec.run.output_hidden_states_mean.shape)
print("span shapes:")
for name, arr in rec.span_hidden_states_mean.items():
    print(f"  span_{name}: {arr.shape}")

## 4. Analysis: subject vs. object distance across layers

Do subject and object hidden-state representations diverge at specific layer depths?

In [ ]:
num_layers = records[0].run.num_layers

# Stack subject and object means across all records
# Shape: (num_records, num_layers, hidden)
subj_stack = np.stack([r.span_hidden_states_mean["subject"] for r in records], axis=0)
obj_stack  = np.stack([r.span_hidden_states_mean["object"]  for r in records], axis=0)

print("Layer-averaged L2 distance between subject and object representations:")
for layer in range(num_layers):
    dists = np.linalg.norm(subj_stack[:, layer, :] - obj_stack[:, layer, :], axis=-1)
    bar   = "█" * int(dists.mean() / 10)
    print(f"  Layer {layer:2d}  mean_dist={dists.mean():7.2f}  {bar}")

In [ ]:
# Per-record comparison at the final layer
final_layer = num_layers - 1
print(f"Subject–object cosine similarity at final layer ({final_layer}):")
for rec in records:
    s = rec.span_hidden_states_mean["subject"][final_layer]
    o = rec.span_hidden_states_mean["object"][final_layer]
    cos = np.dot(s, o) / (np.linalg.norm(s) * np.linalg.norm(o) + 1e-9)
    label = rec.properties["label"]
    print(f"  [{label:>10}]  {rec.instance.text[:40]:<42}  cos={cos:.4f}")

## 5. Dump to disk

In [ ]:
outdir = tempfile.mkdtemp(prefix="internals_extraction_")
print(f"Writing to: {outdir}")

dump(
    records,
    outdir,
    save_attentions=True,           # include attention matrices
    save_full_hidden_states=False,  # means only (smaller files)
)

print("Files written:")
for f in sorted(os.listdir(outdir)):
    size = os.path.getsize(os.path.join(outdir, f))
    print(f"  {f:<20} {size / 1024:.1f} KB")

## 6. Load from disk

In [ ]:
arrays_list, metadata_list = load(outdir)

print(f"Loaded {len(arrays_list)} records")
print()

for arrays, meta in zip(arrays_list, metadata_list):
    print(f"  [{meta['index']}] {meta['properties']['label']:>10}")
    print(f"       properties: {meta['properties']}")
    print(f"       spans:      {meta['spans']}")
    print(f"       arrays:     {list(meta['arrays'].keys())}")
    print(f"       input_hs_mean shape: {arrays['input_hidden_states_mean'].shape}")
    print(f"       span_subject shape:  {arrays['span_subject'].shape}")
    print()

## 7. Full hidden states (optional)

Set `save_full_hidden_states=True` to also store the full `(num_layers, seq_len, hidden)` arrays. Useful for attention pattern analysis or probing classifiers over individual token positions.

In [ ]:
outdir_full = tempfile.mkdtemp(prefix="internals_full_")

dump(records, outdir_full, save_full_hidden_states=True)

arrays_list_full, _ = load(outdir_full)
a = arrays_list_full[0]

print("Arrays with save_full_hidden_states=True:")
for key, val in a.items():
    print(f"  {key:<35} {val.shape}")

## 8. Configuration tips

Turn off signals you don't need to reduce memory and processing time.

In [ ]:
# For probing tasks — only hidden states needed
internals_extraction.set_config(
    extract_attentions=False,
    extract_logits=False,
    max_stored_runs=500,    # keep many runs for a full dataset
)
print(internals_extraction.config)

In [ ]:
# Don't forget to restore if you want everything again
internals_extraction.set_config(
    extract_attentions=True,
    extract_logits=True,
    max_stored_runs=10,
)